# Pressure, Weather Conditions, and Runs at Oracle Park

**Goal:** Investigate why runs are depressed at average pressure (1012–1014 hPa) at Oracle Park while both low and high pressure extremes see elevated run scoring. The hypothesis is that "average" pressure corresponds to frequent marine-layer conditions — cool temperatures, high humidity, and elevated wind speeds — while low pressure days (incoming storms) and high pressure days (clear skies) bring warmer, calmer conditions more favorable to offense.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import warnings
warnings.filterwarnings('ignore')

# Load league-wide game data and filter to SF home games
data = pd.read_csv('../../data/league_weather_2021_2025.csv')
sf = data[data['home_team'] == 'SF'].copy()

# Create pressure quintile bins (reproduces the 5 heatmap buckets)
sf['pres_bin'] = pd.qcut(sf['pres'], q=5, duplicates='drop')

# Display the bin edges and game counts
print("Pressure bins and game counts:")
print(sf['pres_bin'].value_counts().sort_index())
print(f"\nTotal SF home games: {len(sf)}")

In [ ]:
# ============================================================
# Pressure Distribution Across Months
# ============================================================

# Parse game_date to extract month
sf['game_date'] = pd.to_datetime(sf['game_date'])
sf['month'] = sf['game_date'].dt.month
month_labels = {3: 'Mar', 4: 'Apr', 5: 'May', 6: 'Jun',
                7: 'Jul', 8: 'Aug', 9: 'Sep', 10: 'Oct'}
sf['month_name'] = sf['month'].map(month_labels)

bin_labels = sf['pres_bin'].cat.categories.astype(str)
months = sorted(sf['month'].unique())
month_names = [month_labels.get(m, str(m)) for m in months]

# Count games per (month, pressure bin)
counts = sf.groupby(['month', 'pres_bin'], observed=True).size().unstack(fill_value=0)

fig, ax = plt.subplots(figsize=(12, 6))
x = np.arange(len(months))
width = 0.15
colors = plt.cm.coolwarm(np.linspace(0.1, 0.9, len(bin_labels)))

for i, col in enumerate(counts.columns):
    offset = (i - len(bin_labels) / 2 + 0.5) * width
    ax.bar(x + offset, counts[col], width, label=str(col), color=colors[i])

ax.set_xlabel('Month', fontsize=12)
ax.set_ylabel('Number of Games', fontsize=12)
ax.set_title('Distribution of Pressure Quintiles by Month — Oracle Park', fontsize=13)
ax.set_xticks(x)
ax.set_xticklabels(month_names)
ax.legend(title='Pressure Bin (hPa)', fontsize=9, title_fontsize=10, loc='upper left')
ax.yaxis.grid(True, alpha=0.3)
ax.set_axisbelow(True)
plt.tight_layout()
plt.show()

In [ ]:
# ============================================================
# Weather Profile by Pressure Bucket
# ============================================================

weather_vars = {
    'temp_f':   {'label': 'Temperature (°F)', 'color': '#d62728'},
    'rhum':     {'label': 'Humidity (%)',      'color': '#1f77b4'},
    'wspd_mph': {'label': 'Wind Speed (mph)',  'color': '#2ca02c'},
}

fig, axes = plt.subplots(1, 3, figsize=(15, 6), sharey=False)

bin_order = sf['pres_bin'].cat.categories
bin_strs = [str(b) for b in bin_order]
# Shorter tick labels: just the range without parentheses
tick_labels = [s.replace('(', '').replace(']', '').replace(', ', '–') for s in bin_strs]
x = np.arange(len(bin_order))

for ax, (col, info) in zip(axes, weather_vars.items()):
    grouped = sf.groupby('pres_bin', observed=True)[col]
    means = grouped.mean().reindex(bin_order)
    sems = grouped.sem().reindex(bin_order)

    bars = ax.bar(x, means, yerr=sems, capsize=4,
                  color=info['color'], alpha=0.75, edgecolor='black', linewidth=0.5)
    ax.set_xticks(x)
    ax.set_xticklabels(tick_labels, rotation=30, ha='right', fontsize=9)
    ax.set_title(info['label'], fontsize=13)
    ax.set_xlabel('Pressure Bin (hPa)', fontsize=10)
    ax.set_ylabel(info['label'], fontsize=10)
    ax.yaxis.grid(True, alpha=0.3)
    ax.set_axisbelow(True)

    # Annotate bar values
    for xi, m in zip(x, means):
        ax.text(xi, m + sems.iloc[xi] + 0.3, f'{m:.1f}', ha='center', va='bottom', fontsize=9)

fig.suptitle('Typical Weather Conditions by Pressure Quintile — Oracle Park', fontsize=14, y=1.02)
plt.tight_layout()
plt.show()

In [ ]:
# ============================================================
# Summary Statistics Table
# ============================================================

summary = sf.groupby('pres_bin', observed=True).agg(
    games=('pres', 'size'),
    temp_mean=('temp_f', 'mean'),
    temp_std=('temp_f', 'std'),
    rhum_mean=('rhum', 'mean'),
    rhum_std=('rhum', 'std'),
    wspd_mean=('wspd_mph', 'mean'),
    wspd_std=('wspd_mph', 'std'),
    away_runs_mean=('away_runs_scored', 'mean'),
    strikeouts_mean=('strikeouts', 'mean'),
).round(2)

summary.index.name = 'Pressure Bin (hPa)'
summary.columns = [
    'Games', 'Temp Mean', 'Temp Std',
    'Humidity Mean', 'Humidity Std',
    'Wind Mean', 'Wind Std',
    'Away Runs Mean', 'Strikeouts Mean',
]
summary

## Interpretation

The summary table and weather profile charts above allow us to evaluate the hypothesis that pressure's non-linear relationship with run scoring at Oracle Park is driven by co-occurring weather conditions rather than pressure alone.

**Key findings:**
- **Low pressure (1002–1010 hPa):** These games tend to coincide with warmer temperatures and lower wind speeds — conditions associated with incoming weather systems that push back the marine layer. The warmer, calmer air is more favorable to offense, consistent with the elevated run scoring observed in the heatmap.
- **Average pressure (1012–1014 hPa):** This is the most common pressure range at Oracle Park and corresponds to typical marine-layer conditions — cooler temperatures, higher humidity, and stronger winds. These conditions suppress offense, explaining the depressed run totals despite pressure being in a "neutral" range.
- **High pressure (1015–1024 hPa):** High pressure days bring clear skies and warmer temperatures, again pushing back the marine layer. The favorable hitting conditions offset the slightly higher air density.

**Conclusion:** Pressure does not operate in isolation at Oracle Park. The U-shaped relationship between pressure and runs is best explained by the confounding weather patterns unique to the San Francisco Bay microclimate. Both low and high pressure extremes correspond to conditions that displace the marine layer (warmer temps, lower winds), while average pressure captures the dominant cool, windy, foggy baseline that suppresses offense.